# Covariance flow and conserved geometry

A small scalar example of the closed Bures flow. We compare the covariance ODE with the actual two-neuron factor ODE and check the support-gap loss envelope. Install the repository package first (see this directory's README); no path manipulation is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from linear_networks import (
    QuadraticLoss, integrate_covariance, integrate_factors,
    psd_square_root, signed_spectrum, support_margins,
)

loss = QuadraticLoss([[1.0]])
sigma0 = np.array([[1.5, -0.2], [-0.2, 0.9]])
times = np.linspace(0.0, 3.0, 201)
path = integrate_covariance(sigma0, loss, times)
# This deterministic factorization has exactly the same initial covariance.
w0 = np.sqrt(2.0) * psd_square_root(sigma0)
factors = integrate_factors(w0, loss, times)
np.testing.assert_allclose(path.covariances, factors.covariances, atol=1e-8, rtol=1e-8)

## Read the invariant leaf

Write $r=\Sigma^{vu}$ and $a=(\Sigma^u+\Sigma^v)/2$. The difference $b=(\Sigma^u-\Sigma^v)/2$ and $a^2-r^2$ are conserved. For unequal diagonal blocks, the projected PSD boundary is $a\geq\sqrt{r^2+b^2}$. This is covariance geometry: it does not require a Gaussian neuron law.

In [ ]:
r = path.predictors[:, 0, 0]
a = np.trace(path.covariances, axis1=1, axis2=2) / 2
b = (path.covariances[:, 0, 0] - path.covariances[:, 1, 1]) / 2
spectrum = np.array([signed_spectrum(s, d_u=1) for s in path.covariances])
np.testing.assert_allclose(b, b[0], atol=1e-8)
np.testing.assert_allclose(spectrum, np.broadcast_to(spectrum[0], spectrum.shape), atol=1e-8)
gap = support_margins(sigma0, d_u=1).gap
bound = path.loss_gaps[0] * np.exp(-2 * loss.kappa * gap * times)
assert np.max(path.loss_gaps - bound) < 1e-8
print(f'Initial support gap: {gap:.6f}; final loss gap: {path.loss_gaps[-1]:.3e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), constrained_layout=True)
r_grid = np.linspace(-0.5, 1.3, 250)
axes[0].plot(r_grid, np.sqrt(r_grid**2 + b[0]**2), color='0.7', label='PSD boundary at fixed b')
axes[0].plot(r_grid, np.sqrt(r_grid**2 + a[0]**2 - r[0]**2), '--', color='0.5', label='invariant leaf')
axes[0].plot(r, a, color='#2458A6', label='covariance flow')
axes[0].scatter([r[0], r[-1]], [a[0], a[-1]], c=['#D1495B', '#00876C'])
axes[0].set(xlabel='predictor r', ylabel='average variance a')
axes[0].legend(fontsize=8)
axes[1].semilogy(times, path.loss_gaps, label='computed loss gap', color='#2458A6')
axes[1].semilogy(times, bound, '--', label='support-gap envelope', color='#D1495B')
axes[1].set(xlabel='time', ylabel='loss gap')
axes[1].legend(fontsize=8)
plt.show()

The two ODE solvers agree up to their tolerances, not bit-for-bit. Change the cross covariance or the two diagonal variances and rerun. The support gap depends on the full covariance. Near rank-deficient boundaries it must be interpreted with numerical care; the assertions here are consistency checks, not proofs.